# End-to-end agent pipeline on Colab

**Architecture**: Setup → Outline → for each chapter [Retriever (RAG) → Writer (LoRA) → Checker → rewrite up to 3 times]

**Prerequisites**: under Drive `tunshi_lora/` you must have:
- `checkpoints/final/` — the LoRA adapter
- `vectordb/` — the RAG vector store

**Expected time**: ~8 minutes to load + 30-50 minutes to generate three chapters (with up to nine rewrites).

## Cell 1 — Mount Drive and verify prerequisites

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
ADAPTER_DIR = '/content/drive/MyDrive/tunshi_lora/checkpoints/final'
VECTORDB_SRC = '/content/drive/MyDrive/tunshi_lora/vectordb'
VECTORDB_LOCAL = '/content/vectordb'

assert os.path.exists(f'{ADAPTER_DIR}/adapter_model.safetensors'), f'❌ LoRA adapter 缺失: {ADAPTER_DIR}'
assert os.path.exists(f'{VECTORDB_SRC}/chroma.sqlite3'), f'❌ vectordb 没上传到 Drive: {VECTORDB_SRC}'

print('✅ Adapter:')
!ls -lh {ADAPTER_DIR}/adapter_model.safetensors

print('\n✅ vectordb (Drive):')
!du -sh {VECTORDB_SRC}

# Copy vectordb to Colab local disk (Drive direct reads are slow)
if not os.path.exists(VECTORDB_LOCAL):
    print(f'\n复制 vectordb 到 Colab 本地（30 秒）...')
    !cp -r {VECTORDB_SRC} {VECTORDB_LOCAL}
print(f'\n✅ vectordb 在 {VECTORDB_LOCAL}')
!ls {VECTORDB_LOCAL}

Mounted at /content/drive
✅ Adapter:
-rw------- 1 root root 155M May 21 18:10 /content/drive/MyDrive/tunshi_lora/checkpoints/final/adapter_model.safetensors

✅ vectordb (Drive):
71M	/content/drive/MyDrive/tunshi_lora/vectordb

复制 vectordb 到 Colab 本地（30 秒）...

✅ vectordb 在 /content/vectordb
a3c2843c-5248-4b66-8e6c-3b57fb733414  chroma.sqlite3


## Cell 2 — Install dependencies

In [2]:
!pip install -q transformers==4.46.3 peft==0.13.2 accelerate==1.0.1 sentence-transformers chromadb
!pip uninstall -y bitsandbytes 2>/dev/null

import transformers, peft, sentence_transformers, chromadb
print(f'transformers:          {transformers.__version__}')
print(f'peft:                  {peft.__version__}')
print(f'sentence-transformers: {sentence_transformers.__version__}')
print(f'chromadb:              {chromadb.__version__}')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 36.2 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 330.9/330.9 kB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 117.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 129.0 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 122.4 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 19.6 MB/s eta 0:00:00
   ━━━

## Cell 3 — Load the LoRA-adapted model (5-8 minutes on first run)

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

BASE_MODEL = 'Qwen/Qwen2.5-7B'

print('[1/3] 加载 tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print('[2/3] 加载 base Qwen2.5-7B (bf16) ...')
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, torch_dtype=torch.bfloat16, trust_remote_code=True,
).to('cuda')

print('[3/3] 套 LoRA adapter ...')
model = PeftModel.from_pretrained(base, ADAPTER_DIR)
model.eval()

print(f'\n✅ 显存: {torch.cuda.memory_allocated()/1024**3:.1f} GB')

[1/3] 加载 tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model-00003-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/3.56G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

[3/3] 套 LoRA adapter ...

✅ 显存: 14.4 GB


## Cell 4 — Load the RAG components (BGE-zh + ChromaDB)

In [4]:
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings

print('[1/2] Loading BAAI/bge-base-zh-v1.5 (first download ~390 MB) ...')
bge = SentenceTransformer('BAAI/bge-base-zh-v1.5', device='cuda')

print('[2/2] Connecting to ChromaDB ...')
client = chromadb.PersistentClient(
    path=VECTORDB_LOCAL,
    settings=Settings(anonymized_telemetry=False))
collection = client.get_collection('tunshi_worldview_v1')
print(f'✅ collection.count() = {collection.count()}')


def rag_search_single(query, top_k=8, source_filter=None, dist_max=0.55):
    """One query → de-duplicated hits below the distance threshold."""
    emb = bge.encode(query, normalize_embeddings=True, convert_to_numpy=True).tolist()
    where = {'source': source_filter} if source_filter else None
    results = collection.query(
        query_embeddings=[emb], n_results=top_k, where=where,
        include=['documents', 'metadatas', 'distances'])
    hits = []
    for doc, meta, dist in zip(results['documents'][0], results['metadatas'][0], results['distances'][0]):
        if dist > dist_max:
            continue
        hits.append({'text': doc, 'source': meta['source'],
                     'raw_line': meta['raw_line'], 'dist': dist})
    return hits


def rag_search(queries, top_k=8, source_filter=None, dist_max=0.55, total_max=20):
    """Multi-query aggregation: run each query, merge, de-dupe by raw_line,
    sort by distance, cap at total_max."""
    if isinstance(queries, str):
        queries = [queries]
    seen = set()
    all_hits = []
    for q in queries:
        for h in rag_search_single(q, top_k=top_k, source_filter=source_filter, dist_max=dist_max):
            key = (h['source'], h['raw_line'])
            if key in seen:
                continue
            seen.add(key)
            all_hits.append(h)
    all_hits.sort(key=lambda x: x['dist'])
    return all_hits[:total_max]


# Smoke test
print('\n=== RAG smoke test ===')
test_hits = rag_search(['stellar-tier cultivation', '武者突破'], top_k=4, total_max=5)
for h in test_hits:
    print(f"  [{h['source']} 行{h['raw_line']} dist={h['dist']:.2f}] {h['text'][:80]}...")


[1/2] 加载 BGE-zh-base-v1.5（首次下载 ~390MB） ...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[2/2] 连 ChromaDB ...
✅ collection.count() = 11630

=== RAG 冒烟测试 ===
  [TSK1 行11902 dist=0.35] **[TSK1, 行 135]**：超级武者 = 国际武者排名前 100...
  [TSK1 行325 dist=0.36] **宇宙级强者能力描述（视频实录）**（新增 [TSK1, 行 18118-18132]）
  - 能在白矮星上生存战斗（地面引力=地球3亿倍，原子会被压碎）
...
  [TSK1 行2680 dist=0.36] **洪**（更新 [TSK1, 行 16676-16678, 16659]）
  - 行星六阶，但拥有自己"领域"，行星级内无敌
  - 武道境界极高，连许多恒...


## Cell 5 — Setup, generation function, helpers

In [5]:
# === Setup ===
# All fields are user-editable. The book title and chapter titles
# are not hardcoded here — they are generated by the Outline agent
# in Cell 7 from this Setup.
SETUP = {
    'protagonist_name': '风灵',
    'time_anchor': '离罗峰出生早亿万纪元',
    'place_anchor': '原始宇宙某处宇宙秘境，此时地球不存在',
    'tier_anchor': '没有限制',
    'golden_finger': '起源大陆的某半浑源传承（主角不知道具体传承内容）',
}

# === LLM generation function ===
def generate(prompt, max_new_tokens=1500, temperature=0.85, top_p=0.9, seed=None):
    if seed is not None:
        torch.manual_seed(seed)
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=3500).to('cuda')
    prompt_len = inputs['input_ids'].shape[1]
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=True, temperature=temperature, top_p=top_p,
            repetition_penalty=1.1, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][prompt_len:], skip_special_tokens=True)

print('✅ Setup + generate() ready')
for k, v in SETUP.items():
    print(f'  {k}: {v}')


✅ Setup + generate() ready
  protagonist_name: 风灵
  time_anchor: 离罗峰出生早亿万纪元
  place_anchor: 原始宇宙某处宇宙秘境，此时地球不存在
  tier_anchor: 没有限制
  golden_finger: 起源大陆的某半浑源传承（主角不知道具体传承内容）


model.safetensors:   0%|          | 0.00/409M [00:00<?, ?B/s]

## Cell 6 — Checker (five hard rules with automatic scoring)

In [6]:
import re

# Cross-universe blacklist (Xueying Lord terms; must never appear in Tunshi fanfic)
XUEYING_WORDS = ['开辟境', '东伯雪鹰', '雷霆领域', '雪鹰',
                 '黑沼', '雷神军', '灵剑山']

# AI fragmentation regex: 3+ consecutive period-terminated lines of <=8 chars
AI_SHORT_RE = re.compile(r'(^[^。\n]{1,8}。\n){3,}', re.MULTILINE)

# Web-novel meta-noise: residue from the LoRA training corpus
#   author notes, reader comments, serialisation markers, solicitation, etc.
NOISE_PATTERNS = [
    r'未完待续', r'还在连载', r'连载中', r'下一章',
    r'求收藏', r'求推荐', r'求订阅', r'求月票', r'求打赏',
    r'感谢.{0,10}打赏', r'感谢.{0,10}书友', r'感谢.{0,10}读者',
    r'作者的话', r'作者注', r'读者.{0,5}说', r'读者评论',
    r'本章完', r'章节内容', r'书友们', r'本书读者群',
    r'ixdzs', r'爱下电', r'txt80',
]
NOISE_RE = re.compile('|'.join(NOISE_PATTERNS))

# Placeholder residue: model echoing back '[XX]' / '【XX】' template scaffolding
# (e.g. '【主角】风灵', '[书名]', '【时空】')
PLACEHOLDER_RE = re.compile(r'[【\[][\u4e00-\u9fff]{1,8}[】\]]')


def checker(text, setup):
    """Eight hard rules. Returns (passed, issues_list, score)."""
    issues = []

    # 1. Cross-universe contamination
    for w in XUEYING_WORDS:
        c = text.count(w)
        if c > 0:
            issues.append(f"Rule 1 (cross-universe): '{w}' appears {c} times")

    # 2. AI-style fragmentation
    m = AI_SHORT_RE.search(text)
    if m:
        issues.append(f"Rule 2 (AI fragmentation): 3+ short period-terminated lines at pos {m.start()}")

    # 3. Protagonist independence
    name = setup['protagonist_name']
    name_count = text.count(name)
    luofeng_count = text.count('罗峰')
    if name_count < 8:
        issues.append(f"Rule 3 (protagonist): '{name}' appears only {name_count} times (< 8)")
    if luofeng_count > 5:
        issues.append(f"Rule 3 (protagonist): canonical '罗峰' appears {luofeng_count} times (> 5)")

    # 4. Web-novel paragraph format
    fw_pairs = text.count('　　')
    lines = text.count('\n')
    ratio = fw_pairs / max(1, lines)
    if ratio < 0.2:
        issues.append(f"Rule 4 (paragraph format): 　　 ratio {ratio:.2f} < 0.2")

    # 5. Chapter length
    if len(text) < 1200:
        issues.append(f"Rule 5 (length): {len(text)} chars < 1200")

    # 6. Meta-noise residue (training corpus leakage)
    noise_hits = NOISE_RE.findall(text)
    if noise_hits:
        # Show up to 5 unique
        unique = list(dict.fromkeys(noise_hits))[:5]
        issues.append(f"Rule 6 (meta-noise): leaked author/reader/serialisation markers: {unique}")

    # 7. Placeholder residue (model echoing template scaffolding)
    ph_hits = PLACEHOLDER_RE.findall(text)
    if ph_hits:
        unique = list(dict.fromkeys(ph_hits))[:5]
        issues.append(f"Rule 7 (placeholder echo): {unique}")

    # 8. Golden finger embodiment
    # Try to extract a few keywords from the golden-finger string (drop common verbs)
    gf = setup.get('golden_finger', '')
    # Pull 2-4 character Chinese keywords from the golden finger text
    gf_keywords = re.findall(r'[\u4e00-\u9fff]{2,4}', gf)
    # Filter out generic verbs / prepositions / conjunctions
    STOP = {'一块', '神秘', '某个', '一种', '一枚', '一颗', '一份', '一团',
            '能在', '可以', '可能', '比如', '或者', '什么', '怎么样',
            '突然', '不能', '不会'}
    gf_keywords = [k for k in gf_keywords if k not in STOP][:6]
    if gf_keywords:
        if not any(k in text for k in gf_keywords):
            issues.append(f"Rule 8 (golden-finger embodiment): none of {gf_keywords} appears in this chapter")

    score = max(0, 100 - len(issues) * 12)
    return len(issues) == 0, issues, score


# self-test
test_text = '　　李夜站在街头。\n　　李夜深吸一口气。\n李夜笑了。\n他不笑。\n他不哭。'
passed, issues, score = checker(test_text, {'protagonist_name': '李夜', 'golden_finger': '一块紫色晶石'})
print(f'Checker self-test: passed={passed}, score={score}')
for i in issues: print(f'  ❌ {i}')


Checker 测试: passed=False, score=60
  ❌ 主角名「风灵」只出现 0 次（< 8，主角戏份不足）
  ❌ 章节字数偏少（36 < 1200）


## Cell 7 — Setup agent + Outline agent

In [ ]:
import time, re

# === RAG: multi-angle retrieval (covers tier, golden finger, place, era) ===
rag_queries = [
    f'{SETUP["tier_anchor"]} 修炼 突破',
    f'{SETUP["golden_finger"]}',
    f'{SETUP["place_anchor"]}',
    f'{SETUP["time_anchor"]} 历史',
    f'{SETUP["protagonist_name"]} 主角',
    '武者 境界 体系',
]
hits = rag_search(rag_queries, top_k=6, total_max=24)
rag_context = '\n'.join(
    f"- [{h['source']} 行{h['raw_line']}] {h['text'][:200]}" for h in hits[:18]
)
print(f'=== RAG retrieved {len(hits)} unique chunks across {len(rag_queries)} queries ===')
print(rag_context[:600])
print('...')

# === Outline generation ===
# IMPORTANT: do NOT put bracketed placeholders like [书名] / [章节名] inside the
# template — the model will copy them verbatim. We use trailing-prompt
# continuation instead: end with "书名：《" so the model fills in directly.

print('\n=== Generating outline (book title + 3 chapter titles + plot beats) ===')

# Hint to the model whether the setup is far from the canonical Luo Feng era.
# If the user wrote anything suggesting "before Luo Feng" / "far future" /
# "another universe" / "亿万纪元", soften the RAG context as historical reference
# rather than current state.
era_hints = ['亿万纪元', '亿年前', '远古', '上古', '另一', '平行', '之前', '之后']
era_remote = any(h in (SETUP['time_anchor'] + SETUP['place_anchor']) for h in era_hints)

remote_clause = ''
if era_remote:
    remote_clause = (
        '\n⚠️ 本作时空 远离 原著《吞噬星空》主线时代——请把上面的世界观参考'
        '视为「历史背景或宇宙级常识」，不要让原著的人物（罗峰、洪、雷神、坐山客、'
        '巴巴塔、宇宙尊者、九剑尊者等具名角色）作为本作主要人物出现。\n'
    )

outline_prompt = f"""请为《吞噬星空》衍生宇宙写一部 3 章原创小说大纲。

【主角设定（必须严格遵守）】
- 主角姓名：{SETUP["protagonist_name"]}
- 起始境界：{SETUP["tier_anchor"]}
- 时空背景：{SETUP["time_anchor"]} · {SETUP["place_anchor"]}
- 金手指（核心驱动）：{SETUP["golden_finger"]}

【创作要求】
1. 三章必须围绕「金手指」展开：第1章首次激活/感知它，第2章用它解决一个具体冲突，第3章因它招致更大格局。
2. 章节名 2-4 个中文字，体现各章主题，不要叫《序章》《楔子》。
3. 书名 4-8 个中文字，反映「{SETUP["protagonist_name"]}」与金手指的关系，不可与已有原著重名。
4. 每章 4 个情节点：场景 + 冲突 + 主角动作 + 章末小钩子。
5. 主角周围出现的角色全部是你新造的人物，不要套用原著角色名。{remote_clause}

【参考世界观（来自 RAG 检索的注解，含具体行号引用）】
{rag_context[:2000]}

---

请直接输出，不要任何前言。开始：

书名：《"""

t0 = time.time()
outline_body = generate(outline_prompt, max_new_tokens=1400, temperature=0.75)
outline = '书名：《' + outline_body  # prepend the continuation seed for parsing
print(outline)
print(f'\n[took {time.time()-t0:.1f}s]')

# === Parse book title and chapter titles ===
m_book = re.search(r'书名[:：]\s*《\s*([^》\n]+?)\s*》', outline)
book_title = m_book.group(1).strip() if m_book else f'{SETUP["protagonist_name"]}传'

m_chapters = re.findall(r'第\s*[1-3一二三]\s*章\s+([^\n]+)', outline)
chapter_titles = []
for raw in m_chapters[:3]:
    clean = re.split(r'情节点|[:：]|\s{2,}', raw, maxsplit=1)[0].strip()
    chapter_titles.append(clean if clean else f'第{len(chapter_titles)+1}章')
while len(chapter_titles) < 3:
    chapter_titles.append(f'第{len(chapter_titles)+1}章')

# Defend against placeholder residue 【XX】 [XX] in the parsed values
def strip_placeholder(s):
    return re.sub(r'[【\[][^】\]]{0,8}[】\]]', '', s).strip() or s
book_title = strip_placeholder(book_title)
chapter_titles = [strip_placeholder(t) or f'第{i+1}章' for i, t in enumerate(chapter_titles)]

BOOK_META = f'<book_meta>类别=D1 书名={book_title} 作者=AI</book_meta>'

print(f'\n=== Parsed from outline ===')
print(f'  Book title:     《{book_title}》')
print(f'  Chapter titles: {chapter_titles}')
print(f'  BOOK_META used by Writer: {BOOK_META}')


=== RAG retrieved 10 chunks ===
- [TSK2 行23603] **[TSK2, 行 18552-18564]** [type:tier] [id:tier_xiulian_liangdalei] **起源大陆修炼体系两大类**: ①浑源血脉/浑源材料/浑源之力修炼→实力更强但危险（浑源血脉最危险）→传承资源昂贵 ②纯粹大道修行→资源需求少不会失控但实力普遍逊色→须悟出自己的道。"借助浑源血脉修炼，是最危险的"
- [TSK2 行30174] **[TSK2, 行 46972-46998]** [type:concept] [id:concept_shijie_shengming_cai] **世界级浑源生命材料**: 神帝能以一份"世界级浑源生命材料"为修炼核心就已经很不错；凑集几份就可能跳出樊笼；罗峰在起源大陆采集两份（梦域暗金独角+天柱界黑色骨头）；三位异族+两国始祖都培养后辈帮他们采集
- [TSK2 行23545] **[TSK2, 行 12192-12196]** [type:tier] **法则一脉无限神体混沌主宰**: 起源大陆有不少，但两大古国不在意；若是皇族内有此天才则全力栽培。帝楚遇距完整混沌法

=== Generating outline (book title + 3 chapter titles + plot beats) ===
……

请详细描述每个情节点，并说明其对剧情的影响程度。

书名：【主角】风灵

第1章 天才陨落
【主角】风灵：我叫风灵，是一名刚刚通过行星级考核的天才，今天是我的生日，也是我的大喜之日！
第2章 惊艳表现
【主角】风灵：我在这次考核中表现非常出色，得到了很多人的认可与赞赏。
第3章 联谊宴席
【主角】风灵：今天晚上，我将举办一场联欢宴席，邀请了许多志趣相投的朋友一起庆祝。
第4章 宴席风波
【主角】风灵：宴会进行得十分顺利，大家相互交流着彼此的兴趣爱好，气氛十分融洽。
第5章 风波再起
【主角】风灵：然而，就在宴会即将结束的时候，突然有人提出了一些令人尴尬的问题，让整个宴会陷入了一片混乱之中。
第6章 解释与解释
【主角】风灵：面对众人的质疑，我只能尽力去解释，希望能够化解这场风波。
第7章 危机四伏
【主角】风灵：可是，随着时间的推移，事情的发展却越来越超出了我的控制范围

## Cell 8 — Writer + Checker rewrite loop (adversarial core)

In [8]:
MAX_RETRIES = 3


def write_chapter_with_retry(chapter_idx, chapter_title, prev_chapter_tail='', max_retries=MAX_RETRIES):
    """Generate one chapter + Checker verification + rewrite on failure.

    Returns (best_text, attempts_log).
    """
    # Multi-query RAG for this specific chapter
    ch_queries = [
        f'{SETUP["tier_anchor"]} 战斗',
        f'{SETUP["golden_finger"]}',
        f'{chapter_title}',
        f'{SETUP["place_anchor"]}',
    ]
    hits = rag_search(ch_queries, top_k=4, total_max=10)
    ch_rag = '\n'.join(
        f"- [{h['source']} 行{h['raw_line']}] {h['text'][:150]}" for h in hits[:8]
    )

    # Hard-ban canonical character residue when the user's setup is far from
    # the Luo Feng era. Same heuristic as Cell 7.
    era_hints = ['亿万纪元', '亿年前', '远古', '上古', '另一', '平行', '之前', '之后']
    era_remote = any(h in (SETUP['time_anchor'] + SETUP['place_anchor']) for h in era_hints)
    canonical_bans = ['罗峰', '洪', '雷神', '坐山客', '巴巴塔', '九剑尊者',
                      '紫月始祖', '宇宙至尊', '宇宙尊者', '元', '黑沼']
    ban_clause = ''
    if era_remote:
        ban_clause = (
            f'\n本章绝对不能出现以下原著人物或角色名（{SETUP["time_anchor"]}远离原著时代）：\n'
            f'  {"、".join(canonical_bans)}\n'
        )

    best = {'text': '', 'score': -1, 'issues': []}
    attempts_log = []

    for attempt in range(1, max_retries + 1):
        feedback = ''
        if attempt > 1 and best['issues']:
            feedback = (
                '\n【上次写作存在以下问题，本次必须避免】\n'
                + '\n'.join('  - ' + i for i in best['issues'])
                + '\n'
            )

        prompt = f"""{BOOK_META}
<chapter_title>第{chapter_idx}章 {chapter_title}</chapter_title>

【主角与设定（不可违背）】
- 主角：{SETUP["protagonist_name"]}，{SETUP["tier_anchor"]}
- 时空：{SETUP["time_anchor"]} · {SETUP["place_anchor"]}
- 金手指：{SETUP["golden_finger"]}（**本章必须有具体情节体现这个金手指**）

【参考世界观（仅用作背景常识，不要照搬人物名）】
{ch_rag}
{ban_clause}{feedback}
【前章末尾衔接】
{prev_chapter_tail[-300:] if prev_chapter_tail else '（本章是开篇）'}

【写作要求】
- 番茄式短段落（中文全角双空格缩进，对话独立成段），约 2500 中文字。
- 主角名「{SETUP["protagonist_name"]}」至少出现 8 次。
- 本章必须围绕「金手指」展开至少一段具体描写。
- 不要出现「未完待续」「下一章」「作者的话」「求收藏」「读者评论」等同人文套话。
- 不要使用方括号或全角括号包裹的占位符（如「【主角】」「[书名]」）。

直接开始写正文，不要给前言或总结：

　　"""

        temp = 0.85 + 0.05 * (attempt - 1)
        seed = 42 + attempt

        t0 = time.time()
        text = generate(prompt, max_new_tokens=1800, temperature=temp, seed=seed)
        elapsed = time.time() - t0

        passed, issues, score = checker(text, SETUP)

        attempts_log.append({
            'attempt': attempt, 'temperature': temp, 'seed': seed,
            'score': score, 'issues': issues, 'elapsed': elapsed, 'length': len(text),
        })

        print(f'  [Attempt {attempt}] temp={temp:.2f}, score={score}/100, {len(text)} chars, {elapsed:.0f}s')
        for i in issues:
            print(f'    ❌ {i}')

        if score > best['score']:
            best = {'text': text, 'score': score, 'issues': issues}

        if passed:
            print(f'  ✅ Attempt {attempt} passed!')
            break

    if best['score'] < 100:
        print(f'  ⚠️ Chapter {chapter_idx} kept best-of-{max_retries} with {len(best["issues"])} residual issues')
    return best['text'], attempts_log


print('✅ write_chapter_with_retry() ready')


✅ write_chapter_with_retry() 就绪


## Cell 9 — Run end to end: generate three chapters

**Expected time: 30-50 minutes** (up to 3 rewrites per chapter, ~50 s per attempt).

In [9]:
chapters = []
all_logs = []
prev_tail = ''

# chapter_titles was produced by the Outline agent in Cell 7.
# We do not hardcode them.
assert 'chapter_titles' in globals() and len(chapter_titles) == 3, \
    'chapter_titles must come from Cell 7; re-run Cell 7 first.'

global_t0 = time.time()

for idx, title in enumerate(chapter_titles, 1):
    print(f'\n{"=" * 70}')
    print(f'>>> Chapter {idx}: 《{title}》')
    print(f'{"=" * 70}')
    text, log = write_chapter_with_retry(idx, title, prev_tail)
    chapters.append({'idx': idx, 'title': title, 'text': text, 'log': log})
    all_logs.append(log)
    prev_tail = text[-500:]

total_elapsed = time.time() - global_t0
print(f'\n{"=" * 70}')
print(f'All 3 chapters generated. Total time: {total_elapsed/60:.1f} min')
print(f'{"=" * 70}')



>>> Chapter 1: 《天才陨落》
  [Attempt 1] temp=0.85, score=100/100, 3006 字, 138s
  ✅ Attempt 1 通过！

>>> Chapter 2: 《惊艳表现》
  [Attempt 1] temp=0.85, score=60/100, 2878 字, 137s
    ❌ 雪鹰跨源词「雪鹰」出现 1 次（绝对禁止）
    ❌ 主角名「风灵」只出现 0 次（< 8，主角戏份不足）
  [Attempt 2] temp=0.9, score=100/100, 2815 字, 137s
  ✅ Attempt 2 通过！

>>> Chapter 3: 《联谊宴席》
  [Attempt 1] temp=0.85, score=80/100, 3195 字, 138s
    ❌ 主角名「风灵」只出现 0 次（< 8，主角戏份不足）
  [Attempt 2] temp=0.9, score=80/100, 3163 字, 138s
    ❌ 主角名「风灵」只出现 0 次（< 8，主角戏份不足）
  [Attempt 3] temp=0.95, score=100/100, 2957 字, 139s
  ✅ Attempt 3 通过！

All 3 chapters generated. Total time: 13.8 min


## Cell 10 — Print the three chapters

In [10]:
for c in chapters:
    print('=' * 70)
    print(f'第{c["idx"]}章 《{c["title"]}》  ({len(c["text"])} 字)')
    print('=' * 70)
    print(f'　　{c["text"]}')
    print()

第1章 《天才陨落》  (3006 字)
　　　　陈渊站在自己的房间里，打开门走了出去。

　　　　外面，是一个巨大的空间，比之前那个广场还要大上很多倍，大概有 1 公里长宽高。不过陈渊现在还没有办法感受这空间的全部范围。只是能感受到它十分巨大而已。

　　　　“这个就是域外战场？”
　　　　陈渊抬头看着上方的天穹。
　　　　天空中仿佛是一片星空一般，有着无数闪烁着微弱光芒的星星在天上漂浮着。
　　　　陈渊好奇地走向那颗最耀眼的星辰。“这是什么？”他发现那颗星星正朝着自己快速移动过来。

　　　　“看！那是谁？”

　　　　突然，一道声音响起，接着便看到一个人影正在往这边跑来。

　　　　陈渊顺着那道人影望去，只见他正朝一个方向飞去，并且速度非常快，在空中划出一道残影。

　　　　当那人来到近前时，陈渊才发现原来是一位穿着银色战甲的男子，他的脸上带着淡淡的笑容，看上去很是亲切。

　　　　“你好，我叫艾尔多。”银发男子自我介绍道，“欢迎来到我们的阵营。”

　　　　陈渊微微一笑：“我是新来的战士。”

　　　　艾尔多点点头：“我知道了。你先跟着我走吧，我会带你熟悉一下环境。”

　　　　接下来，他们沿着一条宽阔的道路往前走去，很快就来到了一片开阔地带。

　　　　那里有一个巨大的广场，周围还有许多建筑物，包括宿舍楼、餐厅等等设施齐全。

　　　　艾尔多指着四周说道：“这是我们基地内部的主要区域之一——生活区。”
　　随着两人继续前进，又看到了一些士兵们正在进行各种训练活动。

　　“这就是我们平时进行战斗的地方！”艾尔多兴奋地说道，“你看那边有几个朋友正在切磋呢！要不要我们也去试试看？”
　　陈渊摇摇头：“不了，我还是想先了解一下这里的基本情况再说。”

　　　　随后，艾尔多向陈渊详细介绍了基地内的各项设施以及注意事项等信息。

　　　　最后，他还特意提醒了一下：“在这里一定要小心谨慎哦！虽然说我们这边是比较安全的阵营，但是毕竟身处敌对阵营之间，难免会有一些突发状况发生。一旦遇到危险情况，请立刻通过通讯设备通知上级指挥官。”

　　　　听到这话后，陈渊心中暗自警惕起来。
　　不过他也明白，作为一名新人，听从老同志们的建议是非常重要的事情。

　　于是，他就认真地点点头表示明白了。
　　就这样，二人一路聊着天，慢慢地来到了一座精致的小楼前面停了下来。

　　　　艾尔多

## Cell 11 — Final report and save to Drive

Run this after Cell 9 finishes. The LoRA adapter is ~200-250 MB; the outputs are written to Drive under `tunshi_lora/inference_runs/run_001/` and can be downloaded from there.

In [11]:
import json, os

# === Final report ===
print('=' * 70)
print('Adversarial Checker — final report')
print('=' * 70)
print(f'Book title: 《{book_title}》  (generated by the Outline agent)\n')
for c in chapters:
    log = c['log']
    final_score = log[-1]['score']
    n_attempts = len(log)
    n_issues = len(log[-1]['issues'])
    print(f'Chapter {c["idx"]} 《{c["title"]}》:')
    print(f'  attempts: {n_attempts}/3')
    print(f'  final score: {final_score}/100')
    print(f'  remaining issues: {n_issues}')
    if log[-1]['issues']:
        for i in log[-1]['issues']:
            print(f'    - {i}')
    print()

# === Save outputs to Drive ===
out_dir = '/content/drive/MyDrive/tunshi_lora/inference_runs/run_001'
os.makedirs(out_dir, exist_ok=True)

with open(f'{out_dir}/setup.json', 'w') as f:
    json.dump(SETUP, f, ensure_ascii=False, indent=2)
with open(f'{out_dir}/outline.txt', 'w') as f:
    f.write(f'书名：{book_title}\n\n{outline}')
for c in chapters:
    with open(f'{out_dir}/chapter_{c["idx"]}_{c["title"]}.txt', 'w') as f:
        f.write(c['text'])
with open(f'{out_dir}/check_log.json', 'w') as f:
    json.dump({
        'book_title': book_title,
        'setup': SETUP,
        'chapter_logs': [c['log'] for c in chapters],
    }, f, ensure_ascii=False, indent=2)

print(f'✅ saved to: {out_dir}')
!ls -lh {out_dir}/


Adversarial Checker — final report
Book title: 《【主角】风灵》  (generated by the Outline agent)

Chapter 1 《天才陨落》:
  attempts: 1/3
  final score: 100/100
  remaining issues: 0

Chapter 2 《惊艳表现》:
  attempts: 2/3
  final score: 100/100
  remaining issues: 0

Chapter 3 《联谊宴席》:
  attempts: 3/3
  final score: 100/100
  remaining issues: 0

✅ saved to: /content/drive/MyDrive/tunshi_lora/inference_runs/run_001
total 31K
-rw------- 1 root root 8.3K May 22 23:37 chapter_1_天才陨落.txt
-rw------- 1 root root 8.1K May 22 23:37 chapter_2_惊艳表现.txt
-rw------- 1 root root 7.8K May 22 23:37 chapter_3_联谊宴席.txt
-rw------- 1 root root 1.9K May 22 23:37 check_log.json
-rw------- 1 root root 3.4K May 22 23:37 outline.txt
-rw------- 1 root root  289 May 22 23:37 setup.json
